# 28 · Theory — Transactions, Concurrency & Isolation

Module 15 showed `COMMIT`/`ROLLBACK`. This module is the *theory* every backend
engineer needs: what ACID really guarantees, what goes wrong when transactions
run **concurrently**, the standard **isolation levels**, and how SQLite
specifically behaves.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## ACID, precisely
- **Atomicity** — all statements in a transaction succeed or none do. Enforced by
  the rollback journal / WAL: on failure, changes are undone.
- **Consistency** — a transaction moves the database from one valid state to
  another, respecting all constraints (PK, FK, `CHECK`, `NOT NULL`).
- **Isolation** — concurrent transactions don't step on each other; the result is
  *as if* they ran in some serial order.
- **Durability** — once committed, data survives a crash (fsync to disk / WAL
  checkpoint).

## The concurrency anomalies
Isolation exists to prevent these phenomena, which occur when transactions
interleave:

| Anomaly | What happens |
|---------|--------------|
| **Dirty read** | T2 reads data T1 wrote but hasn't committed (and may roll back). |
| **Non-repeatable read** | T1 reads a row twice and gets different values because T2 updated & committed in between. |
| **Phantom read** | T1 re-runs a range query and new rows *appear* because T2 inserted matching rows. |
| **Lost update** | T1 and T2 both read a value, both write; one write silently overwrites the other. |
| **Write skew** | Two transactions read overlapping data and make disjoint writes that together violate an invariant. |

## The ANSI isolation levels
Higher levels prevent more anomalies but reduce concurrency. Databases let you
trade correctness for throughput per transaction.

| Level | Dirty read | Non-repeatable | Phantom |
|-------|-----------|----------------|---------|
| READ UNCOMMITTED | possible | possible | possible |
| READ COMMITTED | prevented | possible | possible |
| REPEATABLE READ | prevented | prevented | possible |
| SERIALIZABLE | prevented | prevented | prevented |

Two implementation strategies:
- **Pessimistic (locking):** take locks so conflicting access blocks. Simple, but
  contention and deadlocks.
- **Optimistic (MVCC — multi-version concurrency control):** readers see a
  consistent *snapshot* and never block writers; conflicts are detected at commit.
  PostgreSQL and MySQL/InnoDB use MVCC.

## How SQLite does it
SQLite is deliberately simple and always effectively **SERIALIZABLE**:
- **Database-level locking** (not row-level). In the classic *rollback journal*
  mode, a writer takes an exclusive lock — readers and the writer can't overlap.
- **WAL mode** (`PRAGMA journal_mode=WAL`) is the big upgrade: a **single writer
  and many readers run concurrently**, because readers see a consistent snapshot
  while the writer appends to a write-ahead log.
- Transaction start modes: `BEGIN DEFERRED` (default; lock acquired lazily on
  first write), `BEGIN IMMEDIATE` (take the write lock now), `BEGIN EXCLUSIVE`.
- When a lock can't be obtained you get **`SQLITE_BUSY` ("database is locked")**;
  `PRAGMA busy_timeout = ms` makes it wait instead of failing immediately.

Let's *prove* SQLite gives no dirty reads, using two live connections.

In [ ]:
import os, tempfile, sqlite3

path = tempfile.mktemp(suffix='.db')
A = sqlite3.connect(path, isolation_level=None)   # manual transaction control
A.execute("PRAGMA journal_mode=WAL")              # readers + 1 writer concurrently
A.execute("CREATE TABLE acct(id INTEGER PRIMARY KEY, balance INTEGER)")
A.execute("INSERT INTO acct VALUES (1, 100)")

B = sqlite3.connect(path, isolation_level=None)   # a second, independent connection

A.execute("BEGIN")                                # A starts a transaction...
A.execute("UPDATE acct SET balance = 999 WHERE id = 1")   # ...and writes (uncommitted)

# B reads WHILE A's change is uncommitted -> must still see the OLD value (no dirty read)
print("B sees during A's open txn :", B.execute("SELECT balance FROM acct WHERE id=1").fetchone()[0])
A.commit()
print("B sees after A committed   :", B.execute("SELECT balance FROM acct WHERE id=1").fetchone()[0])

A.close(); B.close(); os.remove(path.replace('.db','.db'))
for ext in ('-wal','-shm'):
    if os.path.exists(path+ext): os.remove(path+ext)

You should see `100` during A's open transaction and `999` after commit — SQLite never exposed the dirty, uncommitted value.

## Demonstrating `SQLITE_BUSY`
Two write transactions can't overlap — the second gets "database is locked".

In [ ]:
import os, tempfile, sqlite3

path = tempfile.mktemp(suffix='.db')
A = sqlite3.connect(path, isolation_level=None)
A.execute("PRAGMA journal_mode=WAL")
A.execute("CREATE TABLE t(id INTEGER PRIMARY KEY, v INTEGER)")
A.execute("INSERT INTO t VALUES (1, 0)")
B = sqlite3.connect(path, isolation_level=None)

A.execute("BEGIN IMMEDIATE")            # A grabs the write lock
A.execute("UPDATE t SET v = 1 WHERE id = 1")
try:
    B.execute("BEGIN IMMEDIATE")        # B wants to write too -> blocked
    print("B acquired the write lock (unexpected)")
except sqlite3.OperationalError as e:
    print("B blocked with:", e)         # 'database is locked'
A.commit()                              # releasing the lock lets B proceed

A.close(); B.close()
if os.path.exists(path): os.remove(path)
for ext in ('-wal','-shm'):
    if os.path.exists(path+ext): os.remove(path+ext)

## Practical guidance
- Turn on **WAL** for apps with concurrent readers and a writer.
- Set a **`busy_timeout`** so transient locks retry instead of erroring.
- Keep transactions **short** — hold locks for as little time as possible.
- Use `BEGIN IMMEDIATE` when you know you'll write, to fail fast on contention
  rather than mid-transaction.
- On other databases, pick the **isolation level** deliberately: `READ COMMITTED`
  (Postgres default) is usually fine; step up to `SERIALIZABLE` when correctness
  under concurrency is critical.

## Practice

**✏️ Exercise 1.** Which journal mode lets one writer and many readers work at the same time? Confirm the *current* journal mode of this database.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
PRAGMA journal_mode;

### ✅ Recap
ACID defines the guarantees; dirty / non-repeatable / phantom reads, lost updates
and write skew are what isolation prevents; the ANSI levels trade concurrency for
safety via locking or MVCC; and SQLite is serializable with database-level
locking, made concurrent-friendly by WAL.

**Next:** `29_theory_storage_indexes_and_optimizer.ipynb`.